# Vietnam House-Price Prediction

Standalone learning notebook for the second required application.

# Assignment 02: Three Intelligent Applications

This notebook implements the **learning stage** for the three required applications:

1. Diabetes classification using `diabetes.csv` and the tutorial workflow.
2. Vietnam house-price regression using `vietnam_housing_dataset.csv`.
3. Customer-support satisfaction/interest classification using `Customer_support_data.csv`.

Each application follows: **inspect -> clean -> represent -> split -> train -> evaluate -> persist -> inference test**.
All preprocessing is fitted inside a scikit-learn `Pipeline` to prevent data leakage.

## 0. Imports and reproducibility

In [5]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 30)
print('Working directory:', os.getcwd())

Working directory: e:\jupyter project


## 2. Application 2 - Vietnam house-price prediction

**Problem:** predict `Price` from property characteristics. This is regression, so lower MAE/RMSE and higher $R^2$ are desirable. `Address` is transformed into a coarse `City` feature; missing numerical and categorical values are handled only inside the pipeline.

In [6]:
housing = pd.read_csv('vietnam_housing_dataset.csv')
housing['City'] = housing['Address'].fillna('Unknown').str.split(',').str[-1].str.strip().str.rstrip('.')
housing = housing.drop_duplicates().copy()

HOUSE_NUMERIC_FEATURES = ['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms']
HOUSE_CATEGORICAL_FEATURES = [
    'City', 'House direction', 'Balcony direction', 'Legal status', 'Furniture state'
]
HOUSE_TARGET = 'Price'
housing_model = housing[HOUSE_NUMERIC_FEATURES + HOUSE_CATEGORICAL_FEATURES + [HOUSE_TARGET]].dropna(subset=[HOUSE_TARGET])
X_house = housing_model[HOUSE_NUMERIC_FEATURES + HOUSE_CATEGORICAL_FEATURES]
y_house = housing_model[HOUSE_TARGET]
print('Shape after removing duplicate rows and missing targets:', housing_model.shape)
display(housing_model.head())
print('Feature matrix shape before encoding:', X_house.shape)

Shape after removing duplicate rows and missing targets: (30229, 12)


,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,City,House direction,Balcony direction,Legal status,Furniture state,Price
0,84.0,NaN,NaN,4.0,NaN,NaN,Hưng Yên,NaN,NaN,Have certificate,NaN,8.60
1,60.0,NaN,NaN,5.0,NaN,NaN,Hưng Yên,NaN,NaN,NaN,NaN,7.50
2,90.0,6.0,13.0,5.0,NaN,NaN,Hưng Yên,Đông - Bắc,Đông - Bắc,Sale contract,NaN,8.90
3,54.0,NaN,3.5,2.0,2.0,3.0,Hồ Chí Minh,Tây - Nam,Tây - Nam,Have certificate,Full,5.35
4,92.0,NaN,NaN,2.0,4.0,4.0,Hồ Chí Minh,Đông - Nam,Đông - Nam,Have certificate,Full,6.90


Feature matrix shape before encoding: (30229, 11)


In [7]:
X_house_train, X_house_test, y_house_train, y_house_test = train_test_split(
    X_house, y_house, test_size=0.20, random_state=RANDOM_STATE
)

house_numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
house_categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
house_preprocessor = ColumnTransformer([
    ('numeric', house_numeric_pipeline, HOUSE_NUMERIC_FEATURES),
    ('categorical', house_categorical_pipeline, HOUSE_CATEGORICAL_FEATURES),
])

house_models = {
    'Linear Regression': __import__('sklearn.linear_model', fromlist=['LinearRegression']).LinearRegression(),
    'Ridge Regression': Ridge(alpha=10.0),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=18, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=RANDOM_STATE),
}
house_pipelines = {}
house_results = []
for name, estimator in house_models.items():
    pipeline = Pipeline([('preprocessor', house_preprocessor), ('model', estimator)])
    pipeline.fit(X_house_train, y_house_train)
    prediction = pipeline.predict(X_house_test)
    house_pipelines[name] = pipeline
    house_results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_house_test, prediction),
        'RMSE': mean_squared_error(y_house_test, prediction) ** 0.5,
        'R2': r2_score(y_house_test, prediction),
    })
house_results = pd.DataFrame(house_results).set_index('Model').sort_values('RMSE')
display(house_results.round(4))

,MAE,RMSE,R2
Model,,,
Random Forest,1.2275,1.5964,0.4773
Gradient Boosting,1.2779,1.6173,0.4635
Linear Regression,1.4491,1.8102,0.3280
Ridge Regression,1.4515,1.8121,0.3266


### House-price evaluation and representation experiment

The main metrics are MAE, RMSE, and $R^2$. The representation experiment compares a model using the extracted `City` feature with a model that excludes it, testing whether location information improves prediction.

In [8]:
house_best_name = house_results.index[0]
house_best_pipeline = house_pipelines[house_best_name]
print('Selected model by RMSE:', house_best_name)

official_house_features = HOUSE_NUMERIC_FEATURES + HOUSE_CATEGORICAL_FEATURES
no_city_features = HOUSE_NUMERIC_FEATURES + [feature for feature in HOUSE_CATEGORICAL_FEATURES if feature != 'City']
no_city_preprocessor = ColumnTransformer([
    ('numeric', house_numeric_pipeline, HOUSE_NUMERIC_FEATURES),
    ('categorical', house_categorical_pipeline, [feature for feature in HOUSE_CATEGORICAL_FEATURES if feature != 'City']),
])
no_city_pipeline = Pipeline([
    ('preprocessor', no_city_preprocessor),
    ('model', RandomForestRegressor(n_estimators=150, max_depth=18, random_state=RANDOM_STATE, n_jobs=-1)),
])
no_city_pipeline.fit(X_house_train[no_city_features], y_house_train)
city_experiment = pd.DataFrame({
    'Representation': ['With City', 'Without City'],
    'RMSE': [
        mean_squared_error(y_house_test, house_best_pipeline.predict(X_house_test)) ** 0.5,
        mean_squared_error(y_house_test, no_city_pipeline.predict(X_house_test[no_city_features])) ** 0.5,
    ],
})
display(city_experiment.round(4))

with open('house_price_model.sav', 'wb') as file:
    pickle.dump(house_best_pipeline, file)
print('Saved house_price_model.sav')

Selected model by RMSE: Random Forest


,Representation,RMSE
0,With City,1.5964
1,Without City,1.8305


Saved house_price_model.sav
